# Run All Experiments — End-to-End Pipeline

Executes all 4 experiments sequentially on CIFAR-10 using a T4 GPU:

| # | Experiment | Script | Est. time |
|---|---|---|---|
| 1 | Baseline CNN | `run_baseline.py` | ~50 min |
| 2 | HPO (Optuna) | `run_hpo.py` | ~2 h |
| 3 | Evolutionary NAS | `run_evolutionary_nas.py` | ~1.5h |
| 4 | DARTS | `run_darts_search.py` | ~2 h |

**Total: ~40 minutes on GPU.**

### Auto-backup to Google Drive

Checkpoints and results are automatically copied to `My Drive/GGSN-project/` after each experiment. If the Colab runtime disconnects, re-run this notebook and it will **skip completed experiments** by detecting existing files on Drive.

Run with `Runtime > Change runtime type > GPU`.

In [ ]:
import os
from pathlib import Path
import shutil
import time

REPO_URL = "https://github.com/iwadas/GGSN-project.git"
BRANCH = "main"
REPO_DIR = Path("/content/GGSN-project")

if not Path("pyproject.toml").exists():
    os.chdir("/content")
    if not REPO_DIR.exists():
        !git clone -b {BRANCH} {REPO_URL}
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_BACKUP = Path("/content/drive/MyDrive/GGSN-project")
DRIVE_CKPT = DRIVE_BACKUP / "checkpoints"
DRIVE_RESULTS = DRIVE_BACKUP / "results"
DRIVE_PLOTS = DRIVE_BACKUP / "plots"

for d in [DRIVE_CKPT, DRIVE_RESULTS, DRIVE_PLOTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Backup directory: {DRIVE_BACKUP}")

In [ ]:
def backup_file(src: str, dst_dir: Path) -> None:
    p = Path(src)
    if p.exists():
        dst = dst_dir / p.name
        shutil.copy2(p, dst)
        print(f"  [BACKUP] {p.name} -> {dst}")

def backup_dir(src_dir: str, dst_dir: Path) -> None:
    for p in Path(src_dir).iterdir():
        if p.is_file():
            shutil.copy2(p, dst_dir / p.name)

def is_done(checkpoint_name: str) -> bool:
    return (DRIVE_CKPT / checkpoint_name).exists()

## Install Dependencies

In [ ]:
%pip install -q uv
!uv pip install --system -q optuna numpy pandas matplotlib pyyaml tqdm

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

---
## 1. Baseline CNN

Trains the manually-designed baseline CNN. Saves `checkpoints/baseline_cnn.pt`.

In [ ]:
CKPT = "baseline_cnn.pt"

if is_done(CKPT):
    print(f"[SKIP] {CKPT} already exists on Drive, restoring...")
    shutil.copy2(DRIVE_CKPT / CKPT, Path("checkpoints") / CKPT)
else:
    start = time.time()
    print("=" * 50)
    print("Experiment 1/4: Baseline CNN")
    print("=" * 50)

    try:
        !python scripts/run_baseline.py --config experiments/baseline_cnn.yaml
        if Path(f"checkpoints/{CKPT}").exists():
            print("\n[OK] Checkpoint saved")
        else:
            print("\n[WARN] Checkpoint not found, check output above")
    except Exception as e:
        print(f"[FAIL] Baseline CNN failed: {e}")

    print(f"Time: {(time.time() - start)/60:.1f} min")

backup_file(f"checkpoints/{CKPT}", DRIVE_CKPT)
backup_file("results/baseline_summary.json", DRIVE_RESULTS)
backup_file("results/baseline_training_log.csv", DRIVE_RESULTS)
backup_file("plots/baseline_training_curves.png", DRIVE_PLOTS)

---
## 2. Hyperparameter Optimization (Optuna)

Tunes CNN hyperparameters with Optuna, retrains the best config. Saves `checkpoints/hpo_best_baseline_cnn.pt`.

In [ ]:
CKPT = "hpo_best_baseline_cnn.pt"

if is_done(CKPT):
    print(f"[SKIP] {CKPT} already exists on Drive, restoring...")
    shutil.copy2(DRIVE_CKPT / CKPT, Path("checkpoints") / CKPT)
else:
    start = time.time()
    print("=" * 50)
    print("Experiment 2/4: HPO with Optuna")
    print("=" * 50)

    try:
        !python scripts/run_hpo.py --config experiments/hpo_baseline.yaml
        if Path(f"checkpoints/{CKPT}").exists():
            print("\n[OK] Checkpoint saved")
        else:
            print("\n[WARN] Checkpoint not found, check output above")
    except Exception as e:
        print(f"[FAIL] HPO failed: {e}")

    print(f"Time: {(time.time() - start)/60:.1f} min")

backup_file(f"checkpoints/{CKPT}", DRIVE_CKPT)
backup_file("results/hpo_summary.json", DRIVE_RESULTS)
backup_file("results/hpo_trials.csv", DRIVE_RESULTS)
backup_file("results/hpo_best_params.json", DRIVE_RESULTS)
backup_file("results/hpo_best_training_log.csv", DRIVE_RESULTS)
backup_file("plots/hpo_optimization_history.png", DRIVE_PLOTS)
backup_file("plots/hpo_best_training_curves.png", DRIVE_PLOTS)

---
## 3. Evolutionary NAS

Evolves CNN architectures with hardware-aware fitness, retrains the best genome. Saves `checkpoints/evolutionary_best_cnn.pt`.

In [ ]:
CKPT = "evolutionary_best_cnn.pt"

if is_done(CKPT):
    print(f"[SKIP] {CKPT} already exists on Drive, restoring...")
    shutil.copy2(DRIVE_CKPT / CKPT, Path("checkpoints") / CKPT)
else:
    start = time.time()
    print("=" * 50)
    print("Experiment 3/4: Evolutionary NAS")
    print("=" * 50)

    try:
        !python scripts/run_evolutionary_nas.py --config experiments/evolutionary_nas.yaml
        if Path(f"checkpoints/{CKPT}").exists():
            print("\n[OK] Checkpoint saved")
        else:
            print("\n[WARN] Checkpoint not found, check output above")
    except Exception as e:
        print(f"[FAIL] Evolutionary NAS failed: {e}")

    print(f"Time: {(time.time() - start)/60:.1f} min")

backup_file(f"checkpoints/{CKPT}", DRIVE_CKPT)
backup_file("results/evolutionary_summary.json", DRIVE_RESULTS)
backup_file("results/evolutionary_population.csv", DRIVE_RESULTS)
backup_file("results/evolutionary_best_genome.json", DRIVE_RESULTS)
backup_file("results/evolutionary_best_training_log.csv", DRIVE_RESULTS)
backup_file("results/evolutionary_pareto_frontier.csv", DRIVE_RESULTS)
backup_file("plots/evolutionary_progress.png", DRIVE_PLOTS)
backup_file("plots/evolutionary_best_training_curves.png", DRIVE_PLOTS)
backup_file("plots/evolutionary_accuracy_vs_latency.png", DRIVE_PLOTS)
backup_file("plots/evolutionary_accuracy_vs_parameters.png", DRIVE_PLOTS)
backup_file("plots/evolutionary_pareto_frontier.png", DRIVE_PLOTS)

---
## 4. DARTS Differentiable Search

Runs differentiable architecture search, derives discrete architecture, retrains. Saves `checkpoints/darts_best_cnn.pt`.

In [ ]:
CKPT = "darts_best_cnn.pt"

if is_done(CKPT):
    print(f"[SKIP] {CKPT} already exists on Drive, restoring...")
    shutil.copy2(DRIVE_CKPT / CKPT, Path("checkpoints") / CKPT)
else:
    start = time.time()
    print("=" * 50)
    print("Experiment 4/4: DARTS")
    print("=" * 50)

    try:
        !python scripts/run_darts_search.py --config experiments/darts_search.yaml
        if Path(f"checkpoints/{CKPT}").exists():
            print("\n[OK] Checkpoint saved")
        else:
            print("\n[WARN] Checkpoint not found, check output above")
    except Exception as e:
        print(f"[FAIL] DARTS failed: {e}")

    print(f"Time: {(time.time() - start)/60:.1f} min")

backup_file(f"checkpoints/{CKPT}", DRIVE_CKPT)
backup_file("results/darts_summary.json", DRIVE_RESULTS)
backup_file("results/darts_alpha_log.csv", DRIVE_RESULTS)
backup_file("results/darts_derived_genome.json", DRIVE_RESULTS)
backup_file("results/darts_best_training_log.csv", DRIVE_RESULTS)
backup_file("plots/darts_alpha_convergence.png", DRIVE_PLOTS)
backup_file("plots/darts_best_training_curves.png", DRIVE_PLOTS)

---
## Results Summary

In [ ]:
import json

results = {}
for name, path in [
    ("Baseline CNN", "results/baseline_summary.json"),
    ("HPO (Optuna)", "results/hpo_summary.json"),
    ("Evolutionary NAS", "results/evolutionary_summary.json"),
    ("DARTS", "results/darts_summary.json"),
]:
    p = Path(path)
    if p.exists():
        data = json.loads(p.read_text())
        results[name] = data

print(f"{'Experiment':<20} {'Accuracy':>9} {'Params':>10} {'Latency (ms)':>12}")
print("-" * 55)
for name, data in results.items():
    acc = data.get("test_accuracy", data.get("accuracy", "?"))
    params = data.get("parameters", "?")
    lat = data.get("latency_ms", "?")
    if isinstance(acc, float):
        acc = f"{acc*100:.2f}%" if acc < 1 else f"{acc:.2f}%"
    if isinstance(params, int):
        params = f"{params/1e3:.0f}K"
    print(f"{name:<20} {str(acc):>9} {str(params):>10} {str(lat):>12}")

In [ ]:
# Final backup — copy any remaining files to Drive
print("Copying all results and plots to Drive...")
backup_dir("results", DRIVE_RESULTS)
backup_dir("plots", DRIVE_PLOTS)
print()

print("=== Checkpoints on Drive ===")
!ls -lh "$DRIVE_CKPT"

print("\n=== Results on Drive ===")
!ls -lh "$DRIVE_RESULTS"

print("\n=== Plots on Drive ===")
!ls -lh "$DRIVE_PLOTS"

## Files To Keep

All files are also backed up to `My Drive/GGSN-project/`.

### Checkpoints (for ensemble / distillation)
- `checkpoints/baseline_cnn.pt`
- `checkpoints/hpo_best_baseline_cnn.pt`
- `checkpoints/evolutionary_best_cnn.pt`
- `checkpoints/darts_best_cnn.pt`

### Results
- `results/*_summary.json` — accuracy, params, latency
- `results/*_training_log.csv` — per-epoch loss/accuracy
- `results/hpo_trials.csv` — all HPO trials
- `results/evolutionary_population.csv` — all evaluated genomes
- `results/evolutionary_pareto_frontier.csv` — Pareto-optimal points
- `results/darts_alpha_log.csv` — architecture weight evolution

### Plots
- `plots/*.png` — training curves, Pareto frontier, alpha convergence